This script contains the function that can be used for converting Picasso synthetic data to ehtim format so we can reconstruct the image using full pipeline. 
It also contains the function that will convert back the final ehtim reconstructed image to Picasso format (here the format i want is the intensityfield that can be directly used to fit summary statistics)

In [1]:
using Picasso
using DelimitedFiles

In [2]:
config_file = "/home/amandeep-kaur/Downloads/Aspire2025_Picasso-main/src/config.txt"
config = Picasso.read_config(config_file)

Dict{String, Any} with 100 entries:
  "SOURCE_WIDTH"    => 20
  "IMG"             => true
  "AMR"             => false
  "STEPSIZE"        => 0.01
  "IMG_HEIGHT"      => 200
  "NPRIM"           => 8
  "FIT_IMG"         => true
  "GPU_SWITCH"      => false
  "RK45"            => 4
  "INCLINATION"     => 163
  "SOURCE_pa"       => 170.0
  "logscale"        => true
  "NDIM"            => 4
  "max_steps"       => 100000.0
  "RT_OUTER_CUTOFF" => 10000.0
  "PLOT_IMG"        => false
  "SYNCHROTRON"     => true
  "PLOT_SPEC"       => false
  "rcam"            => 100000.0
  ⋮                 => ⋮

In [ ]:
function psize_rad(config::Dict)

    nx = config["IMG_WIDTH"]
    ny = config["IMG_HEIGHT"] 

    src_dist_rg = config["SOURCE_DIST"] * Picasso.SPEED_OF_LIGHT^2 / (Picasso.GGRAV * config["MBH"])
   
    # Angular size of each pixel (rad/pixel)
    psizex = config["CAM_SIZE_IMG"] / (nx * src_dist_rg)
    psizey = config["CAM_SIZE_IMG"] / (ny * src_dist_rg)

    return src_dist_rg, psizex, psizey

end

In [ ]:

function deg_to_hms(degrees::Float64)
    total_hours = degrees / 15
    hours = floor(Int, total_hours)
    
    total_minutes = (total_hours - hours) * 60
    minutes = floor(Int, total_minutes)
    
    seconds = (total_minutes - minutes) * 60

    return hours, minutes, seconds
end

In [ ]:

function Picasso_to_ehtim(config::Dict; n0 = 1)

    rad_to_muas(rad) = rad * 180.0 * 3.6e9 / pi
    muas_to_rad(muas) = muas * pi / (180.0 * 3.6e9)


    imgfile =  ("/home/amandeep-kaur/aspire-black-hole-project/Data_files/Synthetic_data_for_res/img_data_$(Int(n0)).dat")
    img_data = readdlm(imgfile)[:, 1:7]
    img_ehtim = open(
        "/home/amandeep-kaur/aspire-black-hole-project/Data_files/Picasso_2_ehtim/img_ehtim_$(Int(n0)).dat"
        , "w")
    nx = config["IMG_WIDTH"]
    ny = config["IMG_HEIGHT"] 
   
    # Angular size of each pixel (rad/pixel)
    src_dist_rg, psizex, psizey = psize_rad(config)

    x_as = (img_data[:, 1] .- (nx / 2)) * rad_to_muas(psizex) * 1e-6
    y_as = (img_data[:, 2] .- (ny / 2)) * rad_to_muas(psizey) * 1e-6

    data_len = length(x_as)
    max_I=maximum(img_data[:, 3])
    
    x0c = 0.0
    y0c = 0.0

    # Observation time
    mjd = config["obs_time"]

    # RA, DEC and position angle (M87 values)
    ra = config["SOURCE_RA"]
    dec = config["SOURCE_DEC"]

    ra_h, ra_m, ra_s = deg_to_hms(ra)
    dec_h, dec_m, dec_s = deg_to_hms(dec)

    pa = 0.0

    # Frequency
    freq = config["IMG_FREQ"]

    # Source name
    source = "M87"

    println(img_ehtim, "# SRC: $source\n# RA: $ra_h h $ra_m m $ra_s s\n# DEC: $dec_h h $dec_m m $dec_s s\n# MJD: $mjd\n# RF: 230 GHz\n# FOVX: $nx pix $(rad_to_muas(psizex) * 1e-6 * nx) as\n# FOVY: $ny pix $(rad_to_muas(psizey) * 1e-6 * ny) as\n# ------------------------------------\n# x (as)       y (as)       I (jy/pixel) Q (jy/pixel) U (jy/pixel)")
    println(img_data[1, 2])
    for i in 1:Int(sqrt(data_len))
        for j in 1:data_len
            if Int(img_data[j, 2]) == (i - 1)
                println(img_ehtim, "$(round(x_as[j], digits = 10)) $(round(y_as[j], digits = 10)) $(round(img_data[j, 3] / maximum(img_data[:, 3]), digits = 10)) 0.0 0.0")
                # println(img_ehtim, "$(round(x_as[j], digits = 10)) $(round(y_as[j], digits = 10)) $(round(img_data[nx*ny-j+1, 3] / maximum(img_data[:, 3]), digits = 10)) 0.0000000000 0.0000000000")
            end
        end
    end
 
    close(img_ehtim)                     

end

In [ ]:
function ehtim_to_Picasso(config::Dict; n0 = 1)

    rad_to_muas(rad) = rad * 180.0 * 3.6e9 / pi
    muas_to_rad(muas) = muas * pi / (180.0 * 3.6e9)

    
    # ehtimfile = "/home/amandeep-kaur/Documents/Outputs_ehtim_pipeline/blur_$(Int(n0)).dat"
    ehtimfile = "/home/amandeep-kaur/Documents/Outputs_ehtim_pipeline/blur200.txt"
    # "/home/amandeep-kaur/Documents/Outputs_ehtim_pipeline/blur$(Int(n0)).dat"
    img_data_P = readdlm(ehtimfile)[:, 1:3]
    # Extract ehtim data columns
    x_as = img_data_P[:, 1]
    y_as = img_data_P[:, 2]
    Ival = img_data_P[:, 3]

   
    nx = config["IMG_WIDTH"]   
    ny = config["IMG_HEIGHT"]  

    # Pixel size (rad/pixel)
    src_dist_rg, psizex, psizey = psize_rad(config)
    println("psizex is given by $psizex and $psizey")
    # Convert to pixel coordinates
    x_pix = x_as ./ (rad_to_muas(psizex)*(1e-6)) .+ (nx/2)
    y_pix = y_as ./ (rad_to_muas(psizey)*(1e-6)) .+ (ny/2)

    # Normalize intensities
    maxI = 2.9919465087237833e19
      #maximum(Ival)
    println("The value is $maxI")
    I_pix = Ival *maxI
   
   
    out_file = "/home/amandeep-kaur/Documents/Outputs_ehtim_pipeline/e_2_P$(Int(n0)).dat"
    img_Picasso = open(out_file, "w")

    # Header
    # println(img_Picasso, "#  x_pix   y_pix    I_pix    I_pix    0 0 0")

   
    for j in 1:length(x_pix)
    println(img_Picasso,
        "$(round(Int,x_pix[j])) " *
        "$(round(Int,y_pix[j])) " *
        "$(round(I_pix[j], digits = 10)) " *
        "$(round(I_pix[j], digits = 10)) 0 0 0"
    )
      

    end


    close(img_Picasso)                     
    println("ehtim_to_Picasso file: $out_file")

end
